# Week 5 Exercise — Expert Knowledge Worker for West Africa / Côte d'Ivoire

**Author:** asket  
**Location:** `community-contributions/asket/`

Full RAG pipeline aligned with Week 5: chunking (RecursiveCharacterTextSplitter, metadata `doc_type`), embeddings (OpenAI or HuggingFace), ChromaDB, retriever + LLM, simple evaluation, Gradio. **Bilingual (French or English).** Domains: **agriculture** (cocoa, rice, food security), **health** (malaria, maternal and child health), **finance** (mobile money, microfinance).

---

## Week 5 roadmap

| Day | Topic |
|-----|--------|
| Day 2 | Chunking, embeddings, Chroma |
| Day 3 | Retriever + LLM |
| Day 4 | Evaluation |
| Day 5 | Structured knowledge base |


## Setup — Imports and config

**Dependencies:** langchain-community, langchain-text-splitters, langchain-openai, langchain-chroma, chromadb, gradio, python-dotenv; optional langchain-huggingface for low-cost embeddings.

In [2]:
# Install dependencies (run once per environment)
# On Homebrew Python, use --user --break-system-packages:
%pip install -q langchain-community langchain-text-splitters langchain-openai langchain-chroma langchain-core chromadb gradio python-dotenv --user --break-system-packages
# Optional for low-cost embeddings: %pip install -q langchain-huggingface sentence-transformers --user --break-system-packages

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try brew install
    xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a Python library that isn't in Homebrew,
    use a virtual environment:
    
    python3 -m venv path/to/venv
    source path/to/venv/bin/activate
    python3 -m pip install xyz
    
    If you wish to install a Python application that isn't in Homebrew,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. You can install pipx with
    
    brew install pipx
    
    You may restore the old behavior of pip by passing
    the '--break-system-packages' flag to pip, or by adding
    'break-system-packages = true' to your pip.conf file. The latter
    will permanently disable this error.
    
    If you disable this error, we STRONGLY recommend that you additionally
    pass the '--user' flag to pip, or set 

In [6]:
import os
import glob
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

load_dotenv(override=True)

MODEL = "gpt-4o-mini"
DB_NAME = "vector_db_asket_africa"
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
K_RETRIEVE = 5
USE_HF_EMBEDDINGS = os.environ.get("USE_HF_EMBEDDINGS", "").strip().lower() in ("1", "true", "yes")

/Users/franckasket/Library/Python/3.13/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Part A — Load, chunk, embed, Chroma

Load by category from `knowledge-base-africa/` (or resolved path). RecursiveCharacterTextSplitter(800, 150), metadata `doc_type`. Embeddings: OpenAI or HuggingFace (all-MiniLM-L6-v2) when OPENAI_API_KEY unset or USE_HF_EMBEDDINGS=1.

In [7]:
def get_knowledge_base_path() -> Path:
    """Resolve knowledge-base-africa/ next to notebook or from cwd."""
    candidates = [
        Path("knowledge-base-africa"),
        Path("../knowledge-base-africa"),
        Path("week5/community-contributions/asket/knowledge-base-africa"),
    ]
    for p in candidates:
        if p.exists() and p.is_dir():
            return p
    return Path("knowledge-base-africa")

kb_path = get_knowledge_base_path()
print(f"Knowledge base path: {kb_path}")

Knowledge base path: knowledge-base-africa


In [8]:
folders = sorted(glob.glob(str(kb_path / "*")))
folders = [f for f in folders if Path(f).is_dir()]
documents = []
for folder in folders:
    doc_type = Path(folder).name
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
    docs = loader.load()
    for doc in docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents from {len(folders)} categories: {[Path(f).name for f in folders]}")

Loaded 6 documents from 3 categories: ['agriculture', 'finance', 'health']


In [9]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")

Split into 6 chunks (size=800, overlap=150)


In [10]:
openai_key = os.environ.get("OPENAI_API_KEY")
if openai_key and not USE_HF_EMBEDDINGS:
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings()
    print("Using OpenAI embeddings")
else:
    from langchain_huggingface import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    print("Using HuggingFace embeddings (all-MiniLM-L6-v2)")

Using OpenAI embeddings


In [11]:
from langchain_chroma import Chroma

if Path(DB_NAME).exists():
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=DB_NAME)
print(f"Vector store '{DB_NAME}' created with {vectorstore._collection.count()} chunks")

Vector store 'vector_db_asket_africa' created with 6 chunks


## Part B — Retriever + LLM

Retriever k=5. Chat: OpenRouter or OpenAI; system prompt with context; answer in user's language (French or English).

In [12]:
retriever = vectorstore.as_retriever(search_kwargs={"k": K_RETRIEVE})

In [13]:
if os.environ.get("OPENROUTER_API_KEY"):
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(
        model=MODEL,
        temperature=0,
        openai_api_base="https://openrouter.ai/api/v1",
        openai_api_key=os.environ["OPENROUTER_API_KEY"],
    )
    print("Using OpenRouter for chat")
else:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model=MODEL, temperature=0)
    print("Using OpenAI for chat")

Using OpenAI for chat


In [14]:
SYSTEM_PROMPT_TEMPLATE = """You are an expert knowledge assistant for West Africa and Côte d'Ivoire.
You answer questions on agriculture (cocoa, rice, food security), health (malaria, maternal and child health), and finance (mobile money, microfinance).
Use ONLY the context below to answer. If the answer is not in the context, say so.
Respond in the SAME language as the user (French or English).

Context:
{context}
"""

def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [15]:
answer_question("What is Côte d'Ivoire's role in cocoa production?", [])

"Côte d'Ivoire is the world's largest producer of cocoa, accounting for about 40% of global supply. The crop is mainly grown in the forest zones in the south and southwest of the country. Smallholder farmers typically cultivate 2–5 hectares of cocoa."

## Evaluation idea (Day 4)

Simple check: one test question + expected keywords in the answer.

In [16]:
TEST_QUESTION = "What are the main malaria control measures in West Africa?"
EXPECTED_KEYWORDS = ["nets", "LLIN", "ACT", "insecticide", "vaccine"]  # at least one should appear

answer = answer_question(TEST_QUESTION, [])
answer_lower = answer.lower()
found = [k for k in EXPECTED_KEYWORDS if k.lower() in answer_lower]
print(f"Question: {TEST_QUESTION}")
print(f"Answer: {answer[:500]}...")
print(f"Expected keywords found: {found}")

Question: What are the main malaria control measures in West Africa?
Answer: The main malaria control measures in West Africa include long-lasting insecticidal nets (LLINs), indoor residual spraying (IRS), and artemisinin-based combination therapy (ACT)....
Expected keywords found: ['nets', 'LLIN', 'ACT']


## Gradio — ChatInterface

Title and description for West Africa / Côte d'Ivoire; user asks in EN or FR, answer in same language.

In [17]:
def chat_for_gradio(message, history):
    return answer_question(message, history)

demo = gr.ChatInterface(
    fn=chat_for_gradio,
    title="Expert Knowledge Worker — West Africa / Côte d'Ivoire",
    description="Ask questions in English or French on agriculture (cocoa, rice, food security), health (malaria, maternal & child health), and finance (mobile money, microfinance). Answers are based on the curated knowledge base.",
)
demo.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.
